## 1、下载数据集

In [1]:
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

In [2]:
ner_datasets = load_dataset("msra_ner", cache_dir="../data",trust_remote_code=True)

In [3]:
ner_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 45001
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3443
    })
})

In [4]:
print(ner_datasets["train"][0])

{'id': '0', 'tokens': ['当', '希', '望', '工', '程', '救', '助', '的', '百', '万', '儿', '童', '成', '长', '起', '来', '，', '科', '教', '兴', '国', '蔚', '然', '成', '风', '时', '，', '今', '天', '有', '收', '藏', '价', '值', '的', '书', '你', '没', '买', '，', '明', '日', '就', '叫', '你', '悔', '不', '当', '初', '！'], 'ner_tags': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [5]:
ner_datasets["train"].features

{'id': Value(dtype='string', id=None),
 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC'], id=None), length=-1, id=None)}

In [6]:
label_list = ner_datasets["train"].features["ner_tags"].feature.names
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

## 2、数据预处理

In [7]:
# 从HuggingFace加载，输入模型名称，即可加载对于的分词器
tokenizer = AutoTokenizer.from_pretrained('bert-base-chinese', cache_dir='./models')
tokenizer

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


BertTokenizerFast(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [8]:
tokenizer(ner_datasets["train"][0]["tokens"], is_split_into_words=True)   # 对于已经做好tokenize的数据，要指定is_split_into_words参数为True

{'input_ids': [101, 2496, 2361, 3307, 2339, 4923, 3131, 1221, 4638, 4636, 674, 1036, 4997, 2768, 7270, 6629, 3341, 8024, 4906, 3136, 1069, 1744, 5917, 4197, 2768, 7599, 3198, 8024, 791, 1921, 3300, 3119, 5966, 817, 966, 4638, 741, 872, 3766, 743, 8024, 3209, 3189, 2218, 1373, 872, 2637, 679, 2496, 1159, 8013, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [9]:
print(tokenizer(ner_datasets["train"][0]["tokens"], is_split_into_words=True).word_ids())

[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, None]


In [10]:
res = tokenizer("transformers task")
res

{'input_ids': [101, 162, 10477, 8118, 12725, 8755, 8346, 8998, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [11]:
tokenizer("transformers task").word_ids()

[None, 0, 0, 0, 0, 0, 1, 1, None]

In [12]:
res.input_ids

[101, 162, 10477, 8118, 12725, 8755, 8346, 8998, 102]

In [13]:
tokenizer.decode([101, 162, 10477, 8118, 12725, 8755, 8346, 8998, 102])

'[CLS] transformers task [SEP]'

In [14]:
for i in res.input_ids:
    print(tokenizer.decode(i))

[CLS]
t
##ran
##s
##form
##ers
ta
##sk
[SEP]


In [15]:
# 借助word_ids 实现标签映射
def process_function(datasets):
    tokenized_datasets = tokenizer(datasets["tokens"], max_length=128, truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(datasets["ner_tags"]):
        word_ids = tokenized_datasets.word_ids(batch_index=i)
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_id])
        labels.append(label_ids)
    tokenized_datasets["labels"] = labels
    return tokenized_datasets

这段代码是一个处理函数，用于对给定的例子进行标记映射。它接受一个包含文本和相应命名实体识别（NER）标签的字典作为输入，并返回一个包含标记化文本和映射标签的字典。以下是该函数的详细解释：
1. `tokenizer`：这是一个标记器函数，用于将文本转换为标记。它接受文本、最大长度、截断标志和是否按单词分割的标志作为输入。
2. `tokenized_exmaples = tokenizer(examples["tokens"], max_length=128, truncation=True, is_split_into_words=True)`：这行代码使用标记器函数对输入文本进行标记化处理，并将结果存储在`tokenized_exmaples`变量中。最大长度设置为128，如果文本长度超过128，则进行截断。同时，标记器会按单词进行分割。
3. `labels = []`：这是一个空列表，用于存储映射后的标签。
4. `for i, label in enumerate(examples["ner_tags"]):`：这行代码遍历输入字典中的NER标签，并为每个标签分配一个索引`i`。
5. `word_ids = tokenized_exmaples.word_ids(batch_index=i)`：这行代码获取标记化文本中的单词ID。`word_ids`是一个列表，其中包含与每个标记对应的单词ID。如果标记是一个子单词，则其单词ID与其前一个标记的单词ID相同；如果标记是一个特殊标记（如CLS或SEP），则其单词ID为None。
6. `label_ids = []`：这是一个空列表，用于存储映射后的标签ID。
7. `for word_id in word_ids:`：这行代码遍历`word_ids`列表中的每个单词ID。
8. `if word_id is None:`：这行代码检查单词ID是否为None。如果是，说明当前标记是一个特殊标记，我们将标签ID设置为-100。
9. `else:`：如果单词ID不是None，说明当前标记是一个单词的一部分。我们将输入标签中的相应标签ID添加到`label_ids`列表中。
10. `labels.append(label_ids)`：这行代码将`label_ids`列表添加到`labels`列表中。
11. `tokenized_exmaples["labels"] = labels`：这行代码将`labels`列表添加到标记化示例字典中。
12. `return tokenized_exmaples`：这行代码返回包含标记化文本和映射标签的字典。
总之，这个处理函数使用标记器对输入文本进行标记化处理，并将输入标签映射到标记化文本上。映射后的标签将用于后续的命名实体识别任务。


In [16]:
tokenized_datasets = ner_datasets.map(process_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 45001
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3443
    })
})

In [17]:
print(tokenized_datasets["train"][5])

{'id': '5', 'tokens': ['我', '们', '是', '受', '到', '郑', '振', '铎', '先', '生', '、', '阿', '英', '先', '生', '著', '作', '的', '启', '示', '，', '从', '个', '人', '条', '件', '出', '发', '，', '瞄', '准', '现', '代', '出', '版', '史', '研', '究', '的', '空', '白', '，', '重', '点', '集', '藏', '解', '放', '区', '、', '国', '民', '党', '毁', '禁', '出', '版', '物', '。'], 'ner_tags': [0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 4, 0, 0, 0, 0, 0, 0], 'input_ids': [101, 2769, 812, 3221, 1358, 1168, 6948, 2920, 7195, 1044, 4495, 510, 7350, 5739, 1044, 4495, 5865, 868, 4638, 1423, 4850, 8024, 794, 702, 782, 3340, 816, 1139, 1355, 8024, 4730, 1114, 4385, 807, 1139, 4276, 1380, 4777, 4955, 4638, 4958, 4635, 8024, 7028, 4157, 7415, 5966, 6237, 3123, 1277, 510, 1744, 3696, 1054, 3673, 4881, 1139, 4276, 4289, 511, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

## 3、创建模型

In [18]:
# 对于所有的非二分类任务，切记要指定num_labels，否则就会device错误
model = AutoModelForTokenClassification.from_pretrained('bert-base-chinese', cache_dir='./models' , num_labels=len(label_list))


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
model.config.num_labels

7

## 4、创建评估函数

In [20]:
# pip install seqeval
seqeval = evaluate.load("seqeval")
seqeval

EvaluationModule(name: "seqeval", module_type: "metric", features: {'predictions': Sequence(feature=Value(dtype='string', id='label'), length=-1, id='sequence'), 'references': Sequence(feature=Value(dtype='string', id='label'), length=-1, id='sequence')}, usage: """
Produces labelling scores along with its sufficient statistics
from a source against one or more references.

Args:
    predictions: List of List of predicted labels (Estimated targets as returned by a tagger)
    references: List of List of reference labels (Ground truth (correct) target values)
    suffix: True if the IOB prefix is after type, False otherwise. default: False
    scheme: Specify target tagging scheme. Should be one of ["IOB1", "IOB2", "IOE1", "IOE2", "IOBES", "BILOU"].
        default: None
    mode: Whether to count correct entity labels with incorrect I/B tags as true positives or not.
        If you want to only count exact matches, pass mode="strict". default: None.
    sample_weight: Array-like of sha

In [21]:
references = [["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "I-ORG"]]
predictions = [["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "O", "O", "O", "B-ORG", "I-ORG", "O", "O"]]

results = seqeval.compute(predictions=predictions, references=references)


In [22]:
results

{'LOC': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'ORG': {'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 1},
 'PER': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 1},
 'overall_precision': 0.6666666666666666,
 'overall_recall': 0.6666666666666666,
 'overall_f1': 0.6666666666666666,
 'overall_accuracy': 0.8333333333333334}

In [23]:
import numpy as np

def eval_metric(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=-1)

    # 将id转换为原始的字符串类型的标签
    true_predictions = [
        [label_list[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels) 
    ]

    true_labels = [
        [label_list[l] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels) 
    ]

    result = seqeval.compute(predictions=true_predictions, references=true_labels, mode="strict", scheme="IOB2")

    return {
        "f1": result["overall_f1"]
    }
    

这段代码定义了一个评估指标函数`eval_metric`，用于评估序列标注模型在命名实体识别（NER）任务上的性能。函数的输入是一个元组`pred`，其中包含模型预测的分数和真实的标签。代码使用`numpy`库来处理数组运算，并使用`seqeval`库来计算评估指标。
以下是代码的详细解释：
1. `import numpy as np`：导入`numpy`库，用于执行高效的数学运算。
2. `def eval_metric(pred):`：定义一个名为`eval_metric`的函数，它接受一个参数`pred`，这是一个包含模型预测分数和真实标签的元组。
3. `predictions, labels = pred`：将`pred`元组解包为两个变量`predictions`和`labels`，分别存储模型预测的分数和真实的标签。
4. `predictions = np.argmax(predictions, axis=-1)`：使用`numpy`的`argmax`函数沿最后一个轴（即类别轴）找到每个样本的最大预测分数的索引，这些索引对应于预测的标签ID。
5. `true_predictions = [...]`：这是一个列表推导式，用于将预测的标签ID转换为原始的字符串类型的标签。对于每个预测和标签序列，它遍历它们并创建一个新的列表，其中只包含标签不是-100的元素。`label_list[p]`用于将标签ID转换为字符串标签。
6. `true_labels = [...]`：这是另一个列表推导式，与`true_predictions`类似，但它用于转换真实的标签ID为字符串标签。
7. `result = seqeval.compute(predictions=true_predictions, references=true_labels, mode="strict", scheme="IOB2")`：这行代码使用`seqeval`库的`compute`函数来计算评估指标。`true_predictions`是模型的预测，`true_labels`是真实的标签。`mode`参数设置为"strict"，表示严格评估模式，`scheme`参数设置为"IOB2"，表示使用IOB2标签格式。
8. `return {"f1": result["overall_f1"]}`：函数返回一个字典，其中包含整体的F1分数，这是NER任务中常用的评估指标。
总之，这个函数用于评估模型在NER任务上的性能，它将模型输出的分数转换为标签，并使用`seqeval`库来计算F1分数。


## 5、创建训练器

In [24]:
train_args = TrainingArguments(
    output_dir="./models_for_ner",      # 输出文件夹
    per_device_train_batch_size=128,  # 训练时的batch_size
    per_device_eval_batch_size=256,   # 验证时的batch_size
    logging_steps=50,                # log 打印的频率
    evaluation_strategy="epoch",     # 评估策略
    save_strategy="epoch",           # 保存策略
    save_total_limit=3,              # 最大保存数
    num_train_epochs=3,              # 训练轮数, 默认为3
    metric_for_best_model="f1",      # 设定评估指标
    report_to=['tensorboard'],       # tensorboard展示结果
    load_best_model_at_end=True)     # 训练完成后加载最优模型
train_args

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_s

## 6、训练模型

In [25]:
trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=eval_metric,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)
)

In [26]:
trainer.train()

  0%|          | 0/1056 [00:00<?, ?it/s]

{'loss': 0.1749, 'grad_norm': 0.5582171678543091, 'learning_rate': 4.763257575757576e-05, 'epoch': 0.14}
{'loss': 0.0419, 'grad_norm': 0.4679289162158966, 'learning_rate': 4.5265151515151514e-05, 'epoch': 0.28}
{'loss': 0.0329, 'grad_norm': 0.6643112897872925, 'learning_rate': 4.289772727272727e-05, 'epoch': 0.43}
{'loss': 0.0275, 'grad_norm': 0.3979343771934509, 'learning_rate': 4.053030303030303e-05, 'epoch': 0.57}
{'loss': 0.0235, 'grad_norm': 0.3914872705936432, 'learning_rate': 3.816287878787879e-05, 'epoch': 0.71}
{'loss': 0.0257, 'grad_norm': 0.787024736404419, 'learning_rate': 3.579545454545455e-05, 'epoch': 0.85}
{'loss': 0.0222, 'grad_norm': 0.506645679473877, 'learning_rate': 3.342803030303031e-05, 'epoch': 0.99}


  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.022115662693977356, 'eval_f1': 0.9452530754061415, 'eval_runtime': 10.0848, 'eval_samples_per_second': 341.406, 'eval_steps_per_second': 1.388, 'epoch': 1.0}
{'loss': 0.0144, 'grad_norm': 0.5377807021141052, 'learning_rate': 3.106060606060606e-05, 'epoch': 1.14}
{'loss': 0.0123, 'grad_norm': 0.32402661442756653, 'learning_rate': 2.869318181818182e-05, 'epoch': 1.28}
{'loss': 0.0133, 'grad_norm': 0.20607955753803253, 'learning_rate': 2.6325757575757575e-05, 'epoch': 1.42}
{'loss': 0.0116, 'grad_norm': 0.4375413656234741, 'learning_rate': 2.3958333333333334e-05, 'epoch': 1.56}
{'loss': 0.0112, 'grad_norm': 0.2571427524089813, 'learning_rate': 2.1590909090909093e-05, 'epoch': 1.7}
{'loss': 0.0135, 'grad_norm': 0.2939971089363098, 'learning_rate': 1.922348484848485e-05, 'epoch': 1.85}
{'loss': 0.0114, 'grad_norm': 0.31509044766426086, 'learning_rate': 1.6856060606060605e-05, 'epoch': 1.99}


  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.023576583713293076, 'eval_f1': 0.9486003183222544, 'eval_runtime': 10.019, 'eval_samples_per_second': 343.647, 'eval_steps_per_second': 1.397, 'epoch': 2.0}
{'loss': 0.0074, 'grad_norm': 0.1992819756269455, 'learning_rate': 1.4488636363636366e-05, 'epoch': 2.13}
{'loss': 0.0075, 'grad_norm': 0.3247379660606384, 'learning_rate': 1.2121212121212122e-05, 'epoch': 2.27}
{'loss': 0.006, 'grad_norm': 0.23010140657424927, 'learning_rate': 9.75378787878788e-06, 'epoch': 2.41}
{'loss': 0.0063, 'grad_norm': 0.07810487598180771, 'learning_rate': 7.386363636363637e-06, 'epoch': 2.56}
{'loss': 0.0071, 'grad_norm': 0.17341366410255432, 'learning_rate': 5.018939393939394e-06, 'epoch': 2.7}
{'loss': 0.0065, 'grad_norm': 0.13608647882938385, 'learning_rate': 2.651515151515152e-06, 'epoch': 2.84}
{'loss': 0.0059, 'grad_norm': 0.14988873898983002, 'learning_rate': 2.840909090909091e-07, 'epoch': 2.98}


  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.025085777044296265, 'eval_f1': 0.9540046838407494, 'eval_runtime': 10.1423, 'eval_samples_per_second': 339.469, 'eval_steps_per_second': 1.38, 'epoch': 3.0}
{'train_runtime': 1025.5186, 'train_samples_per_second': 131.644, 'train_steps_per_second': 1.03, 'train_loss': 0.02289367010675822, 'epoch': 3.0}


TrainOutput(global_step=1056, training_loss=0.02289367010675822, metrics={'train_runtime': 1025.5186, 'train_samples_per_second': 131.644, 'train_steps_per_second': 1.03, 'total_flos': 8750374708762368.0, 'train_loss': 0.02289367010675822, 'epoch': 3.0})

## 7、评估模型

In [27]:
trainer.evaluate(eval_dataset=tokenized_datasets["test"])

  0%|          | 0/14 [00:00<?, ?it/s]

{'eval_loss': 0.025085777044296265,
 'eval_f1': 0.9540046838407494,
 'eval_runtime': 10.1004,
 'eval_samples_per_second': 340.877,
 'eval_steps_per_second': 1.386,
 'epoch': 3.0}

## 8、预测

In [28]:
from transformers import pipeline

# 使用pipeline进行推理，要指定id2label
model.config.id2label = {idx: label for idx, label in enumerate(label_list)}
model.config

BertConfig {
  "_name_or_path": "bert-base-chinese",
  "architectures": [
    "BertForTokenClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-PER",
    "2": "I-PER",
    "3": "B-ORG",
    "4": "I-ORG",
    "5": "B-LOC",
    "6": "I-LOC"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers": 3,
  "pooler_size_per_head": 128,
  "pooler_type": "first_token_transform",
  "position_embedding_type": "absolute",
  "

In [31]:
ner_pipe = pipeline("token-classification", model=model, tokenizer=tokenizer, device=0)
res = ner_pipe("马云在杭州创建了阿里巴巴")
res

[{'entity': 'B-PER',
  'score': 0.9994667,
  'index': 1,
  'word': '马',
  'start': 0,
  'end': 1},
 {'entity': 'I-PER',
  'score': 0.99928695,
  'index': 2,
  'word': '云',
  'start': 1,
  'end': 2},
 {'entity': 'B-LOC',
  'score': 0.9995776,
  'index': 4,
  'word': '杭',
  'start': 3,
  'end': 4},
 {'entity': 'I-LOC',
  'score': 0.9994968,
  'index': 5,
  'word': '州',
  'start': 4,
  'end': 5},
 {'entity': 'B-ORG',
  'score': 0.99651754,
  'index': 9,
  'word': '阿',
  'start': 8,
  'end': 9},
 {'entity': 'I-ORG',
  'score': 0.9952846,
  'index': 10,
  'word': '里',
  'start': 9,
  'end': 10},
 {'entity': 'I-ORG',
  'score': 0.997056,
  'index': 11,
  'word': '巴',
  'start': 10,
  'end': 11},
 {'entity': 'I-ORG',
  'score': 0.99644715,
  'index': 12,
  'word': '巴',
  'start': 11,
  'end': 12}]

In [29]:
# 指定aggregation_strategy为simple, 返回的ner_pipe为SimplePipeline
ner_pipe = pipeline("token-classification", model=model, tokenizer=tokenizer, device=0, aggregation_strategy="simple")

In [30]:
res = ner_pipe("马云在杭州创建了阿里巴巴")
res

[{'entity_group': 'PER',
  'score': 0.99937683,
  'word': '马 云',
  'start': 0,
  'end': 2},
 {'entity_group': 'LOC',
  'score': 0.9995372,
  'word': '杭 州',
  'start': 3,
  'end': 5},
 {'entity_group': 'ORG',
  'score': 0.9963263,
  'word': '阿 里 巴 巴',
  'start': 8,
  'end': 12}]